# Football Era Lab: the same engine as the browser

This notebook runs the game's own JavaScript engine through Node, so every number here is the number the browser would show. Python only reads the bundled data, starts Node and displays what comes back. Run all cells from the repository root.

You can change the settings below to try any manager, formation and fifteen players, or let the notebook make a seeded draft for you. The browser is where you make every pick and placement yourself.

In [1]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess

import pandas as pd
from IPython.display import display

# R: repository root; G: bundled game data; NODE: executable running the shared engine.
R = Path.cwd()
if not (R / 'data' / 'game.json').exists() and (R.parent / 'data' / 'game.json').exists():
    R = R.parent
if not (R / 'data' / 'game.json').exists():
    raise FileNotFoundError('Run this notebook in the Football Era Lab repository with data/game.json.')
NODE = shutil.which('node')  # Or set an absolute node.exe path here.
if not NODE:
    raise FileNotFoundError('Install Node 20 or later, or set NODE to its executable path.')
G = json.loads((R / 'data' / 'game.json').read_text(encoding='utf-8'))
print('Node:', subprocess.check_output([NODE, '--version'], text=True).strip())
print('Data SHA-256:', hashlib.sha256((R / 'data' / 'game.json').read_bytes()).hexdigest())
print('Data build:', G.get('meta', {}).get('v'))


Node: v24.19.0
Data SHA-256: 4536ee541946b702821fd3d68a1e9c82b19c3bd8115580b01fa40f649fa30ef3
Data build: 2026-10-07


## Where every rating comes from

Each card is one person at one club in one decade. Its rating, its natural positions, its rating in every pitch slot and its six face stats all come from the same source, shown by a code:

| Code | Source |
| --- | --- |
| `f` | EA FIFA / FC, an edition inside the player's stint at this club (FIFA 07 to FC 26) |
| `c` | Championship Manager 01/02 database at this club, put on EA's scale |
| `i` | EA Icon or Hero card |
| `n` | EA, a season within two years of the stint, adjusted for age |
| `m` | Championship Manager, a season within two years of the stint, adjusted for age |
| `e` | Estimated by a model fitted on the EA and CM ratings, where no game database rates the player |

The table counts cards by decade and source. The 1950s to 1970s are still mostly estimates: no openly published game database covers them, and the Championship Manager retro seasons for those years are added as they arrive.

In [2]:
rows = [dict(decade=int(k.split(':')[1]), source=c['s']) for k, Q in G['cards'].items() for c in Q]
display(pd.crosstab(pd.DataFrame(rows).decade, pd.DataFrame(rows).source, margins=True))
catalogue = pd.DataFrame([
    {'manager': m['nm'], 'attack_grade': m['ga'], 'defence_grade': m['gd'], 'formations': ', '.join(m['f'])}
    for m in G['managers']
])
print('Managers:', len(catalogue), '· formations:', ', '.join(G['formations']))
print('Unique people:', len(G['people']), '· club-decades:', len(G['combos']), '· cards:', len(rows))

source,c,e,f,i,m,n,All
decade,,,,,,,
1950,0,2859,0,2,0,0,2861
1960,0,3868,0,8,0,0,3876
1970,0,4795,0,13,0,0,4808
1980,1723,3424,0,37,1035,0,6219
1990,4629,760,0,83,3716,0,9188
2000,2251,3005,2120,94,1940,3053,12463
2010,0,1084,6585,12,76,3213,10970
2020,273,36,1845,0,36,401,2591
All,8876,19831,10550,249,6803,6667,52976


Managers: 96 · formations: 2-3-5, WM, 4-2-4, Catenaccio, 4-4-2, 4-4-2 diamond, 4-3-3, 4-3-3 false nine, 4-2-3-1, 4-3-2-1, 4-3-1-2, 4-1-4-1, 3-5-2, 3-4-3, 3-4-3 diamond, 3-4-2-1, 5-3-2
Unique people: 25382 · club-decades: 910 · cards: 52976


## One player, every slot

A player's strengths and weaknesses decide where he is worth playing. Where a game database rates him, the engine reads his rating in each slot from his card: EA's own per-position ratings, or his Championship Manager attributes put on EA's scale. Playing him in a slot rated below his overall costs exactly that gap. Change `who` to look at someone else.

In [3]:
SL = ['GK', 'LB', 'CB', 'RB', 'LWB', 'RWB', 'CDM', 'CM', 'CAM', 'LM', 'RM', 'LW', 'RW', 'CF', 'ST']
who, at = 'Diego Maradona', 'Napoli'
hits = [(k, c) for k, Q in G['cards'].items() for c in Q
        if G['people'][c['p']]['nm'] == who and at.casefold() in G['clubs'][k.split(':')[0]]['nm'].casefold()]
for k, c in hits:
    print(f"{who}, {at} {k.split(':')[1]}s: rating {c['r']} (source {c['s']}), natural slots {', '.join(c['pos'])}")
    if 'sr' in c:
        display(pd.DataFrame([c['sr']], columns=SL, index=['rating in slot']))
    if 'f6' in c:
        names = ['DIV', 'HAN', 'KIC', 'REF', 'SPD', 'POS'] if c['pos'][0] == 'GK' else ['PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY']
        display(pd.DataFrame([c['f6']], columns=names, index=['face stats']))
    print(f"League apps {c['n']}, goals {c['g']}", '· real stats:', c.get('st', 'none recorded for this card'))

Diego Maradona, Napoli 1980s: rating 91.2 (source c), natural slots CAM, CF, CM

,GK,LB,CB,RB,LWB,RWB,CDM,CM,CAM,LM,RM,LW,RW,CF,ST
rating in slot,14,70,66,71,74,74,75,90,91,89,89,89,89,90,88


,PAC,SHO,PAS,DRI,DEF,PHY
face stats,75,93,96,95,60,83


League apps 161, goals 69 · real stats: none recorded for this card
Diego Maradona, Napoli 1990s: rating 91.2 (source m), natural slots CAM, CF, CM


,GK,LB,CB,RB,LWB,RWB,CDM,CM,CAM,LM,RM,LW,RW,CF,ST
rating in slot,14,70,66,71,74,74,75,90,91,89,89,89,89,90,88


,PAC,SHO,PAS,DRI,DEF,PHY
face stats,75,93,96,95,60,83


League apps 27, goals 12 · real stats: none recorded for this card


## Configure the experiment

Leave both lists empty for a seeded notebook draft. `person_ids` takes fifteen distinct Wikidata person IDs; with IDs alone, the bridge prefers a card from the simulation decade, then the highest rating. `source_cards` keeps exact club-decade cards, for example `{'k': 'Q2641:1980', 'p': 'Q17515'}` for Maradona's Napoli card.

Choose any catalogue formation independently of the manager. `formation=None` uses the manager's first recorded formation.

`placement='best'` lets the engine assign the eleven starters; `placement='ordered'` keeps your order: eleven formation slots, then four substitutes.

`cap=True` drafts under the Salary cap, the browser default: 2 S, 4 A, 4 B, 3 C and 2 D players across all fifteen, by base card rating (S 90 or above, A 85, B 80, C 75, D below 75). `cap=False` is Classic. The cap also applies to fifteen people or cards you supply.

In [4]:
settings = {
    'seed': 20261006,
    'decade': 1990,
    'cap': True,  # Salary cap, as in the browser; False for Classic.
    'manager': None,  # A manager name from the catalogue, or None for a seeded choice.
    'formation': None,  # Any catalogue formation; None uses the manager's first.
    'person_ids': [],
    'source_cards': [],
    'placement': 'best',
}

def engine(settings):
    """Run the repository JavaScript bridge and return its JSON response."""
    p = subprocess.run([NODE, str(R / 'notebooks' / 'engine_bridge.mjs')],
                       input=json.dumps(settings), capture_output=True, text=True,
                       encoding='utf-8', cwd=R, timeout=120, check=False)
    if p.returncode:
        raise RuntimeError(p.stderr.strip() or f'Node exited with code {p.returncode}')
    return json.loads(p.stdout)

result = engine(settings)
print(result['mode'])
print('Rules:', 'Salary cap' if result['cap'] else 'Classic', '· tier counts:', result['tier_counts'])
print('Manager:', result['manager']['nm'], 'Formation:', result['formation'])
team = result.get('team')
if team:
    a, b = team['a'], team['b']
    print('Club spell:', f"{team['club']}, {a}/{(a + 1) % 100:02}–{b}/{(b + 1) % 100:02}")
print('Match parameters:')
print(json.dumps(result['parameters'], indent=2))
display(pd.DataFrame(result['squad']))
print('Source counts:', result['source_counts'])
print('Exact source cards for replay:')
print(json.dumps(result['source_cards']))


Notebook-only seeded draft demonstration: automatic picks, with editable placement.
Rules: Salary cap · tier counts: {'S': 2, 'A': 4, 'B': 4, 'C': 3, 'D': 2}
Manager: Joe Fagan Formation: 4-4-2
Club spell: Liverpool, 1983/84–1984/85
Match parameters:
{
  "be": 0.4115033527449246,
  "ka": 0.5566086036333626,
  "h": 0.2620934744673785,
  "pa": 0.05,
  "fat": 0.04,
  "m1": 60,
  "m2": 75,
  "ns": 3,
  "b": {
    "1950": 1.45864386651915,
    "1960": 1.2789984962526806,
    "1970": 1.185731721121677,
    "1980": 1.176241415929074,
    "1990": 1.1535407576488488,
    "2000": 1.1487314074297343,
    "2010": 1.2313444534429265,
    "2020": 1.2313444534429265
  },
  "src": "Fitted on 2014-19 real club-season results; held out 2020-23. Neutral 2020s scoring level is frozen from training; historical decade levels are measured ratios."
}


,place,id,player,club,card_decade,positions,base_rating,tier,source,tags
0,1: GK,Q11584,Iker Casillas,Real Madrid,2000,GK,90.5,S,f,"{""rock"":1,""tl"":2,""bg"":3}"
1,2: LB,Q189984,Franco Baresi,AC Milan,1990,CB,87.2,A,c,"{""rock"":2,""tl"":1,""bg"":3}"
2,3: CB,Q701934,Ivan Buljan,Hamburger SV,1980,CB,79.8,C,e,{}
3,4: CB,Q295512,Carlos Gamarra,Benfica,1990,CB,82.2,B,m,{}
4,5: RB,Q106625792,Nicolás González Iglesias,Manchester City,2020,"CDM, CM",79.0,C,f,"{""bg"":1}"
5,6: LM,Q363829,Paulo Futre,Benfica,1990,"LW, LM, RW, ST",89.0,A,i,"{""tal"":1,""bg"":1}"
6,7: CM,Q3620472,António Pacheco,Benfica,1990,"LM, LW, CM",83.3,B,m,{}
7,8: CM,Q16056053,Jack Grealish,Manchester City,2020,"LW, LM",84.3,B,f,"{""bg"":1}"
8,9: RM,Q173139,George Weah,AC Milan,1990,"CF, ST",87.2,A,c,"{""tal"":2}"
9,10: ST,Q529207,Ronaldo,Real Madrid,2000,"ST, CF",94.0,S,i,"{""tal"":1,""tl"":1,""poa"":2}"


Source counts: {'f': 4, 'c': 4, 'i': 2, 'm': 2, 'e': 3}
Exact source cards for replay:
[{"k": "Q8682:2000", "p": "Q11584"}, {"k": "Q1543:1990", "p": "Q189984"}, {"k": "Q51974:1980", "p": "Q701934"}, {"k": "Q131499:1990", "p": "Q295512"}, {"k": "Q50602:2020", "p": "Q106625792"}, {"k": "Q131499:1990", "p": "Q363829"}, {"k": "Q131499:1990", "p": "Q3620472"}, {"k": "Q50602:2020", "p": "Q16056053"}, {"k": "Q1543:1990", "p": "Q173139"}, {"k": "Q8682:2000", "p": "Q529207"}, {"k": "Q1543:1990", "p": "Q152844"}, {"k": "Q8682:2000", "p": "Q315043"}, {"k": "Q50602:2020", "p": "Q59381180"}, {"k": "Q51974:1980", "p": "Q829354"}, {"k": "Q51974:1980", "p": "Q508061"}]


## See fit, era and rating changes

The starter table is the engine's own rating breakdown. Position loss comes from the player's slot ratings where his card has them, otherwise from how far the slot is from his natural position. Bench players have no position loss. Links, Timeless and manager grades are game rules.

In [5]:
display(pd.DataFrame(result['starters']))
display(pd.DataFrame(result['bench']))
display(pd.DataFrame([result['lines']]))


,slot,id,player,base_rating,source,card_decade,fit_label,position_loss,era_multiplier,chemistry_points,adjusted_rating
0,GK,Q11584,Iker Casillas,90.5,f,2000,Natural,0.005525,0.99250,0,89.3250
1,LB,Q189984,Franco Baresi,87.2,c,1990,Adapted -7%,0.071101,1.00000,0,81.0000
2,CB,Q701934,Ivan Buljan,79.8,e,1980,Natural,0.000000,0.97000,0,77.4060
3,CB,Q295512,Carlos Gamarra,82.2,m,1990,Natural,0.002433,1.00000,0,82.0000
4,RB,Q106625792,Nicolás González Iglesias,79.0,f,2020,Adapted -6%,0.063291,0.95500,1,71.6700
5,LM,Q363829,Paulo Futre,89.0,i,1990,Natural,0.000000,1.00000,1,90.0000
6,CM,Q3620472,António Pacheco,83.3,m,1990,Natural,0.003601,1.00000,1,84.0000
7,CM,Q16056053,Jack Grealish,84.3,f,2020,Adapted -4%,0.039146,0.95500,1,78.3550
8,RM,Q173139,George Weah,87.2,c,1990,Adapted -5%,0.048165,1.00000,0,83.0000
9,ST,Q529207,Ronaldo,94.0,i,2000,Natural,0.000000,0.99625,0,93.6475


,id,player,base_rating,source,card_decade,position_loss,era_multiplier,adjusted_rating
0,Q315043,César Sánchez,82.6,c,2000,0,0.985,81.361
1,Q59381180,Julián Alvarez,79.0,f,2020,0,0.955,75.445
2,Q829354,Peter Lux,74.9,e,1980,0,0.970,72.653
3,Q508061,Erik Solér,74.8,e,1980,0,0.970,72.556


,attack,midfield,outfield_defence,keeper,keeper_weighted_defence,overall,attack_grade,defence_grade,signature_upgrade,knockout_boost
0,92.697285,83.863268,82.832394,89.325,84.780176,83.503973,B,B,False,0.018974


## Play the league and European Cup

Twenty clubs play home and away; sixteen enter the Cup, with two-legged ties, a neutral final, extra time and penalties. Who scores in each simulated goal follows the players' real scoring rates where their cards have them.

In [6]:
season = result['season']
display(pd.DataFrame(season['L']['tab']))
display(pd.DataFrame([
    {k: v for k, v in fixture.items() if k != 'M'} for fixture in season['L']['res']
]))
cup_rows = []
for round_ties in season['K']['rounds']:
    for t in round_ties:
        cup_rows.append({'round': t['k'], 'home': t['an'], 'away': t['bn'],
                         'aggregate': f"{t['agg'][0]}-{t['agg'][1]}", 'winner': t['wn'],
                         'extra_time': t.get('et', False), 'penalties': t.get('pw')})
display(pd.DataFrame(cup_rows))
print('Cup winner:', season['K']['champN'])
display(pd.DataFrame(result['player_stats']))
if 'awards' in result:
    print(json.dumps(result['awards'], indent=2, ensure_ascii=False))


,id,nm,me,P,W,D,L,GF,GA,Pts
0,Q7156:1990,Barcelona · 1990s,False,38,30,3,5,111,28,93
1,Q1543:1990,AC Milan · 1990s,False,38,27,6,5,114,40,87
2,Q1422:1990,Juventus · 1990s,False,38,25,9,4,109,33,84
3,Q8682:1990,Real Madrid · 1990s,False,38,23,8,7,97,48,77
4,Q15789:1990,Bayern Munich · 1990s,False,38,22,9,7,80,43,75
5,Q132885:1990,Marseille · 1990s,False,38,20,10,8,67,46,70
6,Q1457:1990,Sampdoria · 1990s,False,38,20,8,10,87,63,68
7,Q81888:1990,Ajax · 1990s,False,38,17,9,12,70,58,60
8,Q18656:1990,Manchester United · 1990s,False,38,18,4,16,71,53,58
9,Q180305:1990,Monaco · 1990s,False,38,15,6,17,66,74,51


,rd,h,op,gf,ga
0,1,True,Panathinaikós Athlitikós Ómilos · 1990s,4,0
1,2,NaN,Spartak Moscow · 1990s,1,2
2,3,True,Red Star Belgrade · 1990s,2,0
3,4,NaN,Feyenoord · 1990s,2,1
4,5,True,Dynamo Kyiv · 1990s,3,1
5,6,NaN,Kaiserslautern · 1990s,4,1
6,7,True,PSV Eindhoven · 1990s,3,3
7,8,NaN,Porto · 1990s,1,3
8,9,True,Monaco · 1990s,3,4
9,10,NaN,Ajax · 1990s,0,4


,round,home,away,aggregate,winner,extra_time,penalties
0,Round of 16,Ajax · 1990s,Real Madrid · 1990s,3-7,Real Madrid · 1990s,False,None
1,Round of 16,Kaiserslautern · 1990s,Sampdoria · 1990s,0-4,Sampdoria · 1990s,False,None
2,Round of 16,Monaco · 1990s,Barcelona · 1990s,1-8,Barcelona · 1990s,False,None
3,Round of 16,Dynamo Kyiv · 1990s,Bayern Munich · 1990s,2-3,Bayern Munich · 1990s,False,None
4,Round of 16,Your Era XI,Juventus · 1990s,3-6,Juventus · 1990s,False,None
5,Round of 16,PSV Eindhoven · 1990s,AC Milan · 1990s,1-13,AC Milan · 1990s,False,None
6,Round of 16,Porto · 1990s,Marseille · 1990s,2-3,Marseille · 1990s,False,None
7,Round of 16,Borussia Dortmund · 1990s,Manchester United · 1990s,3-5,Manchester United · 1990s,False,None
8,Quarter-final,AC Milan · 1990s,Real Madrid · 1990s,3-1,AC Milan · 1990s,False,None
9,Quarter-final,Bayern Munich · 1990s,Sampdoria · 1990s,3-2,Bayern Munich · 1990s,False,None


Cup winner: Juventus · 1990s


,id,cl,nm,src,k,g,as,cs,ap,competition
0,Q11584,your-club,Iker Casillas,f,True,0,0,6,37,League
1,Q189984,your-club,Franco Baresi,c,False,0,2,0,35,League
2,Q701934,your-club,Ivan Buljan,e,False,3,2,0,36,League
3,Q295512,your-club,Carlos Gamarra,m,False,0,1,0,37,League
4,Q106625792,your-club,Nicolás González Iglesias,f,False,2,0,0,36,League
...,...,...,...,...,...,...,...,...,...,...
503,Q48330,Q1457:1990,Roberto Mancini,c,False,1,0,0,1,European Cup
504,Q316229,Q15789:1990,Paulo Sérgio,m,False,0,0,0,1,European Cup
505,Q13494,Q15789:1990,Jean-Pierre Papin,c,False,0,0,0,1,European Cup
506,Q336916,Q132885:1990,Abedi Pele,i,False,0,0,0,1,European Cup


{
  "league": {
    "scorer": {
      "id": "Q487155",
      "cl": "Q8682:1990",
      "nm": "Hugo Sánchez",
      "src": "i",
      "k": false,
      "g": 38,
      "as": 2,
      "cs": 0,
      "ap": 37
    },
    "assists": {
      "id": "Q516192",
      "cl": "Q132885:1990",
      "nm": "Franck Sauzée",
      "src": "m",
      "k": false,
      "g": 4,
      "as": 19,
      "cs": 0,
      "ap": 37
    },
    "keeper": {
      "id": "Q215963",
      "cl": "Q7156:1990",
      "nm": "Andoni Zubizarreta",
      "src": "c",
      "k": true,
      "g": 0,
      "as": 0,
      "cs": 21,
      "ap": 36
    },
    "player": {
      "id": "Q483629",
      "cl": "Q1543:1990",
      "nm": "Marco van Basten",
      "src": "c",
      "k": false,
      "g": 38,
      "as": 5,
      "cs": 0,
      "ap": 35
    }
  },
  "cup": {
    "scorer": {
      "id": "Q483629",
      "cl": "Q1543:1990",
      "nm": "Marco van Basten",
      "src": "c",
      "k": false,
      "g": 8,
      "as": 2,
      "cs"

## Replay and keep a result

The same settings, data and engine always give the same season. This cell checks that by running everything twice. Set `export_path` to a folder you keep if you want the full result as JSON; by default nothing is written.

In [7]:
again = engine(settings)
assert result == again, 'The same settings produced a different bridge response.'
print('Deterministic replay matched for these settings and this data bundle.')

export_path = None  # Set a persistent Path to write the full result as JSON.
if export_path is not None:
    p = Path(export_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Wrote:', p.resolve())


Deterministic replay matched for these settings and this data bundle.


## What these numbers can and cannot tell you

Squads come from dated Wikidata club records, so a decade squad can hold players who never shared a season, and apps and goals are spread across the years of each stint. EA and Championship Manager ratings are each game's judgement of a player, put on one scale; the estimates for earlier decades extrapolate a model fitted on 1989-2025 cards. The match model was fitted on real club results from 2014-19 and checked on 2020-23. Era losses, links, tags and manager grades are game rules, not measurements, and every competition uses one modern format.